In [18]:
from mc_experiment import (
    make_seed_counter,
    next_seed,
    standardize_innovations,
    summarize_reference_experiment,
    summarize_mle_augmentation_experiment,
    augmented_config_path,
    size_adjusted_power_comparison,
)

from SymbolicDSGE import ModelParser, DSGESolver, Shock
from SymbolicDSGE.bayesian import make_prior

from numpy import log
import numpy as np

from scipy.stats import chi2, gaussian_kde, norm

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl

from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import cProfile

import contextlib
import io
STDOUT_VOID = lambda: contextlib.redirect_stdout(io.StringIO())

_KNOWN_R = False
_AUGMENTED_PARAM = 'r_coef'
_AUGMENTED_EQUATION = 'Rate'
_AUGMENTED_CONFIG = augmented_config_path(_AUGMENTED_EQUATION)
_MEAS_ERR_SCALE = 0.10
_MC_SAMPLES = 1000
_MC_ALPHA = 0.05
_FIGSIZE_1D = (10, 6)
_FIGSIZE_2D = (12, 6)



In [19]:
# Load reference model
parser = ModelParser("../../MODELS/misspec_test/reference.yaml")
config, kalman = parser.get_all()
solver = DSGESolver(config, kalman)

comp = solver.compile(
    n_state=3,
    n_exog=3,
)
sol = solver.solve(
    comp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

print("Transition matrix:\n", sol.A.round(3), "\n")
print("Shock Loadings:\n", sol.B.round(3))

Transition matrix:
 [[ 0.83   0.     0.     0.     0.   ]
 [ 0.     0.85   0.     0.     0.   ]
 [ 0.288 -0.047  0.28   0.     0.   ]
 [ 0.892  0.708 -1.711  0.     0.   ]
 [ 0.7   -0.115 -1.363  0.     0.   ]] 

Shock Loadings:
 [[ 1.     0.     0.   ]
 [ 0.     1.     0.   ]
 [ 0.     0.     1.   ]
 [ 3.193  0.493 -6.107]
 [ 2.531 -0.406 -4.864]]


In [20]:
# Load Misspecified DGP
parser_dgp = ModelParser("../../MODELS/misspec_test/misspec.yaml")
config_dgp, kalman_dgp = parser_dgp.get_all()
solver_dgp = DSGESolver(config_dgp, kalman_dgp)
comp_dgp = solver_dgp.compile(
    n_state=3,
    n_exog=3,
)
sol_dgp = solver_dgp.solve(
    comp_dgp,
    steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
)

In [21]:
# Large sample simulations used to approximate measurement-noise variances
_large_sample_seed_counter = make_seed_counter(start=100_000)
shocks_large = {
    "g,z": Shock(10_000, "norm", multivar=True, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
    "r": Shock(10_000, "norm", multivar=False, seed=next_seed(_large_sample_seed_counter)).shock_generator(),
}

sim1 = sol_dgp.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

sim2 = sol.sim(
    T=10_000,
    shocks=shocks_large,
    observables=True,
)

In [22]:
T = 200
_plot_seed_counter = make_seed_counter(start=2_000_000)

err_var = np.var(np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]]), axis=0)
mc_reference = summarize_reference_experiment(
    sol,
    sol_dgp,
    T=T,
    err_var=err_var,
    meas_err_scale=_MEAS_ERR_SCALE,
    mc_samples=_MC_SAMPLES,
    known_r=_KNOWN_R,
    alpha=_MC_ALPHA,
)

rep_ref = mc_reference["representative"]
sim_dgp = rep_ref.sim_dgp
obs = rep_ref.obs
kf = rep_ref.kf
std_innov = rep_ref.std_innov
err_scale = rep_ref.err_scale
N, n_obs = kf.innov.shape

_measurement_order = {"OutGap": 0, "Infl": 1, "Rate": 2}
_predictor_order = {"Pi": 0, "x": 1, "r": 2}

def _sort_summary(df):
    out = df.copy()
    if "measurement" in out.columns:
        out["measurement_order"] = out["measurement"].map(_measurement_order)
    if "predictor" in out.columns:
        out["predictor_order"] = out["predictor"].map(_predictor_order)
    if "target" in out.columns:
        out["target_order"] = out["target"].map(_predictor_order)
    if "regressor" in out.columns:
        out["regressor_order"] = out["regressor"].map(_predictor_order)
    sort_cols = [
        col
        for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"]
        if col in out.columns
    ]
    if sort_cols:
        out = out.sort_values(sort_cols)
    return out.drop(columns=[col for col in ["measurement_order", "target_order", "predictor_order", "regressor_order"] if col in out.columns])

sim_ref = sol.sim(
    T=T,
    shocks={
        "g,z": Shock(T, "norm", multivar=True, seed=next_seed(_plot_seed_counter)).shock_generator(),
        "r": Shock(T, "norm", multivar=False, seed=next_seed(_plot_seed_counter)).shock_generator(),
    },
    observables=True,
)
ref = np.column_stack([sim_ref["OutGap"], sim_ref["Infl"], sim_ref["Rate"]])[1:, :]

obs_dgp = np.column_stack([sim1["OutGap"], sim1["Infl"], sim1["Rate"]])[1:, :]
if np.any(err_scale != 0.0):
    _plot_rng = np.random.default_rng(next_seed(_plot_seed_counter))
    obs_dgp = obs_dgp + _plot_rng.normal(scale=np.sqrt(err_scale), size=obs_dgp.shape)

In [23]:
print(f"Known R assumption: {_KNOWN_R}")
print(f"Augmented measurement equation: {_AUGMENTED_EQUATION}")
print(f"Augmented coefficient: {_AUGMENTED_PARAM}")
print(f"Monte Carlo replications: {_MC_SAMPLES}")
print("Noise Covariance:\n", np.diag(err_scale).round(3))

Known R assumption: False
Augmented measurement equation: Rate
Augmented coefficient: r_coef
Monte Carlo replications: 1000
Noise Covariance:
 [[1.208 0.    0.   ]
 [0.    1.679 0.   ]
 [0.    0.    0.078]]


In [24]:
print(f"Monte Carlo Ljung-Box summary across {_MC_SAMPLES} replications:")
display(mc_reference["lb_summary"].round(3))

Monte Carlo Ljung-Box summary across 1000 replications:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,19.197,0.001,0.228,0.0,1000,995,0.995,0.002,0.988,0.998
1,Infl,30.654,0.000,0.272,0.0,1000,1000,1.000,0.000,0.996,1.000
2,Rate,23.303,0.000,0.260,0.0,1000,999,0.999,0.001,0.994,1.000


In [25]:
print(f"Moment Tests summary across {_MC_SAMPLES} replications:")
display(mc_reference["moment_specification_test_summary"].round(3))

Moment Tests summary across 1000 replications:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.126,2.239,0.606,0.002,0.065,0.009,1000,21,0.021,0.005,0.014,0.032,3.0,200,4
1,cov_identity,19.387,203.094,0.000,0.072,1.653,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


In [26]:
print("Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_orthogonalized_summary"]).round(3)

Innovations on orthogonalized predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,Pi,-0.110,-0.006,1.386,-0.083,0.511,0.005,0.042,0.002,0.003,0.030,0.009,0.000,1000,37,0.037,0.006,0.027,0.051
1,OutGap,x,-0.570,-0.236,0.166,-3.438,0.014,0.060,0.005,0.002,0.000,0.032,0.001,0.001,1000,931,0.931,0.008,0.914,0.945
2,OutGap,r,-0.652,-0.023,1.934,-0.321,0.486,0.005,0.061,0.002,0.006,0.031,0.009,0.000,1000,53,0.053,0.007,0.041,0.069
3,Infl,Pi,-1.191,-0.053,1.546,-0.759,0.411,0.008,0.050,0.002,0.003,0.032,0.009,0.000,1000,123,0.123,0.010,0.104,0.145
4,Infl,x,0.103,0.038,0.191,0.537,0.450,0.007,0.006,0.002,0.001,0.033,0.010,0.000,1000,89,0.089,0.009,0.073,0.108
5,Infl,r,-0.579,-0.018,2.162,-0.253,0.478,0.006,0.071,0.002,0.007,0.033,0.009,0.000,1000,63,0.063,0.008,0.050,0.080
6,Rate,Pi,-0.275,-0.064,0.307,-0.906,0.397,0.009,0.009,0.002,0.001,0.031,0.010,0.000,1000,147,0.147,0.011,0.126,0.170
7,Rate,x,0.026,0.048,0.038,0.678,0.424,0.007,0.001,0.002,0.000,0.032,0.009,0.000,1000,100,0.100,0.009,0.083,0.120
8,Rate,r,-0.938,-0.153,0.424,-2.191,0.111,0.028,0.014,0.002,0.001,0.031,0.006,0.001,1000,584,0.584,0.016,0.553,0.614


In [27]:
print("Innovations on raw predicted states (Monte Carlo averages and rejection rates):")
_sort_summary(mc_reference["measurement_regressions_raw_summary"]).round(3)

Innovations on raw predicted states (Monte Carlo averages and rejection rates):


,measurement,predictor,coef,standardized_coef,std_error,t_stat,p_value,r2,mc_se_coef,mc_se_standardized_coef,mc_se_std_error,mc_se_t_stat,mc_se_p_value,mc_se_r2,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
2,OutGap,Pi,-3.577,-0.280,0.865,-4.125,0.002,0.081,0.027,0.002,0.001,0.028,0.000,0.001,1000,994,0.994,0.002,0.987,0.997
1,OutGap,x,-0.562,-0.366,0.100,-5.574,0.000,0.137,0.003,0.002,0.000,0.030,0.000,0.001,1000,1000,1.000,0.000,0.996,1.000
0,OutGap,r,1.605,0.061,1.835,0.870,0.430,0.007,0.045,0.002,0.006,0.024,0.009,0.000,1000,75,0.075,0.008,0.060,0.093
5,Infl,Pi,-0.448,-0.030,1.005,-0.427,0.464,0.006,0.033,0.002,0.001,0.032,0.009,0.000,1000,71,0.071,0.008,0.057,0.089
4,Infl,x,0.010,0.007,0.121,0.101,0.507,0.005,0.004,0.002,0.000,0.032,0.009,0.000,1000,56,0.056,0.007,0.043,0.072
3,Infl,r,-0.786,-0.026,2.053,-0.371,0.470,0.006,0.066,0.002,0.006,0.032,0.009,0.000,1000,67,0.067,0.008,0.053,0.084
8,Rate,Pi,-0.101,-0.037,0.200,-0.521,0.477,0.006,0.006,0.002,0.000,0.030,0.009,0.000,1000,72,0.072,0.008,0.058,0.090
7,Rate,x,0.012,0.035,0.024,0.493,0.473,0.006,0.001,0.002,0.000,0.031,0.009,0.000,1000,76,0.076,0.008,0.061,0.094
6,Rate,r,-0.987,-0.169,0.401,-2.436,0.083,0.033,0.014,0.002,0.001,0.032,0.005,0.001,1000,673,0.673,0.015,0.643,0.701


In [28]:
print("Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_orthogonalized_summary"]).round(3)

Innovation decomposition on orthogonalized predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.447,-1.557,-0.110,-0.110,-0.0,0.0,0.0,0.028,0.021,0.042,0.042,0.0,0.0,0.0
1,OutGap,x,0.046,-0.616,-0.570,-0.570,-0.0,0.0,0.0,0.003,0.003,0.005,0.005,0.0,0.0,0.0
2,OutGap,r,-0.226,-0.427,-0.652,-0.652,-0.0,0.0,0.0,0.039,0.034,0.061,0.061,0.0,0.0,0.0
3,Infl,Pi,-0.043,-1.147,-1.191,-1.191,-0.0,0.0,0.0,0.015,0.048,0.050,0.050,0.0,0.0,0.0
4,Infl,x,0.002,0.101,0.103,0.103,0.0,0.0,0.0,0.002,0.006,0.006,0.006,0.0,0.0,0.0
5,Infl,r,-0.036,-0.543,-0.579,-0.579,0.0,0.0,0.0,0.022,0.069,0.071,0.071,0.0,0.0,0.0
6,Rate,Pi,0.008,-0.283,-0.275,-0.275,0.0,0.0,0.0,0.003,0.009,0.009,0.009,0.0,0.0,0.0
7,Rate,x,-0.000,0.026,0.026,0.026,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.017,-0.921,-0.938,-0.938,0.0,0.0,0.0,0.005,0.014,0.014,0.014,0.0,0.0,0.0


In [29]:
print("Innovation decomposition on raw predicted states (Monte Carlo averages):")
_sort_summary(mc_reference["innovation_decomposition_raw_summary"]).round(3)

Innovation decomposition on raw predicted states (Monte Carlo averages):


,measurement,predictor,beta_measurement_error,beta_state_prediction_error,beta_total_innovation,beta_component_sum,beta_component_gap,abs_beta_component_gap,reconstruction_max_abs_error,mc_se_beta_measurement_error,mc_se_beta_state_prediction_error,mc_se_beta_total_innovation,mc_se_beta_component_sum,mc_se_beta_component_gap,mc_se_abs_beta_component_gap,mc_se_reconstruction_max_abs_error
0,OutGap,Pi,1.777,-5.355,-3.577,-3.577,-0.0,0.0,0.0,0.018,0.013,0.027,0.027,0.0,0.0,0.0
1,OutGap,x,0.182,-0.744,-0.562,-0.562,-0.0,0.0,0.0,0.002,0.002,0.003,0.003,0.0,0.0,0.0
2,OutGap,r,-0.560,2.164,1.605,1.605,-0.0,0.0,0.0,0.044,0.035,0.045,0.045,0.0,0.0,0.0
3,Infl,Pi,-0.025,-0.423,-0.448,-0.448,-0.0,0.0,0.0,0.010,0.031,0.033,0.033,0.0,0.0,0.0
4,Infl,x,-0.002,0.012,0.010,0.010,-0.0,0.0,0.0,0.001,0.004,0.004,0.004,0.0,0.0,0.0
5,Infl,r,-0.034,-0.752,-0.786,-0.786,0.0,0.0,0.0,0.021,0.065,0.066,0.066,0.0,0.0,0.0
6,Rate,Pi,0.006,-0.107,-0.101,-0.101,0.0,0.0,0.0,0.002,0.006,0.006,0.006,0.0,0.0,0.0
7,Rate,x,0.001,0.012,0.012,0.012,0.0,0.0,0.0,0.000,0.001,0.001,0.001,0.0,0.0,0.0
8,Rate,r,-0.014,-0.973,-0.987,-0.987,-0.0,0.0,0.0,0.004,0.014,0.014,0.014,0.0,0.0,0.0


Monte Carlo summaries above aggregate `_MC_SAMPLES` independent draws. The plots and MCMC output below continue on a representative first draw so the original visual workflow remains available.


In [30]:
parser_aug = ModelParser(_AUGMENTED_CONFIG)
config_aug, kalman_aug = parser_aug.get_all()
solver_aug = DSGESolver(config_aug, kalman_aug)
comp_aug = solver_aug.compile(
    n_state=3,
    n_exog=3,
)
priors = {
    _AUGMENTED_PARAM: make_prior(
        'normal',
        parameters={"mean": 0.0, "std": 4.0, "random_state": next_seed(_plot_seed_counter)},
        transform="identity",
    ),
}

with STDOUT_VOID():
    mc_aug = summarize_mle_augmentation_experiment(
        sol,
        solver_aug,
        comp_aug,
        sol_dgp,
        mc_reference,
        T=T,
        candidate_param=_AUGMENTED_PARAM,
        mc_samples=_MC_SAMPLES,
        alpha=_MC_ALPHA,
    )

# estim = lambda: solver_aug.estimate_and_solve(
#     compiled=comp_aug,
#     method="mcmc",
#     n_draws=25_000,
#     burn_in=10_000,
#     thin=2,
#     posterior_point="mean",
#     proposal_scale=1.0,
#     y=obs,
#     priors=priors,
#     steady_state=[0.0, 0.0, 0.0, 0.0, 0.0],
#     random_state=next_seed(_plot_seed_counter),
#     **mc_reference["filter_kwargs"],
# )
# res_aug, sol_aug = estim()

## Diagnostics of the Augmented Model

The figures below still display the representative first draw. The scalar summaries reported in later cells are Monte Carlo averages.


### Marginal LR Test Conditional on $	heta_0$

The table below reports the Monte Carlo MLE summary for the LR test.


In [31]:
print("Monte Carlo LR summary for the MLE-augmented model:")
rep_aug = mc_aug["representative"]
res_mle = rep_aug.res_mle
sol_mle = rep_aug.sol_mle
mle_aug_kf = rep_aug.kf_aug
std_innov_aug_mle = rep_aug.std_innov_aug
sim_aug_mle = rep_aug.sim_aug
mc_aug["lr_summary"].round(3)

Monte Carlo LR summary for the MLE-augmented model:


,estimated_coef,loglik_ref,loglik_aug,lr,p_value,mc_se_estimated_coef,mc_se_loglik_ref,mc_se_loglik_aug,mc_se_lr,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,-0.143,-2909.234,-2905.986,6.495,0.304,0.006,7.266,7.24,0.272,0.012,1000,469,0.469,0.016,0.438,0.5


In [32]:
res_mle

OptimizationResult(kind='mle', x=array([-0.0035022]), theta={'beta': np.float64(0.971), 'kappa': np.float64(0.58), 'tau_inv': np.float64(1.86), 'psi_pi': np.float64(2.19), 'psi_x': np.float64(0.3), 'rho_r': np.float64(0.84), 'rho_g': np.float64(0.83), 'rho_z': np.float64(0.85), 'pi_star': np.float64(3.43), 'r_star': np.float64(3.01), 'sig_r': np.float64(0.18), 'sig_g': np.float64(0.18), 'sig_z': np.float64(0.64), 'rho_gz': np.float64(0.36), 'meas_infl': np.float64(1e-06), 'meas_rate': np.float64(1e-06), 'meas_outgap': np.float64(1e-06), 'meas_rho_ir': np.float64(0.0), 'meas_rho_gi': np.float64(0.0), 'meas_rho_gr': np.float64(0.0), 'Pi_coef': np.float64(0.0), 'x_coef': np.float64(0.0), 'r_coef': np.float64(-0.003502203405043994)}, success=True, message='CONVERGENCE: NORM OF PROJECTED GRADIENT <= PGTOL', fun=np.float64(2506.158799809619), loglik=np.float64(-2506.158799809619), logprior=np.float64(0.0), logpost=np.float64(-2506.158799809619), nfev=8, nit=2, raw=  message: CONVERGENCE: NOR

## Serial Autocorrelation Tests for the Augmented Model

The figure below uses the representative MCMC draw, while the printed table reports Monte Carlo MLE rejection frequencies.


In [33]:
print("Monte Carlo Ljung-Box summary for the MLE-augmented model:")
display(mc_aug["lb_summary"].round(3))

Monte Carlo Ljung-Box summary for the MLE-augmented model:


,measurement,lb_stat,p_value,mc_se_lb_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high
0,OutGap,19.202,0.001,0.228,0.0,1000,995,0.995,0.002,0.988,0.998
1,Infl,30.652,0.000,0.272,0.0,1000,1000,1.000,0.000,0.996,1.000
2,Rate,22.597,0.001,0.259,0.0,1000,998,0.998,0.001,0.993,0.999


In [34]:
print("Reference moment-specification test summary:")
display(mc_reference["moment_specification_test_summary"].round(3))

print("Augmented moment-specification test summary:")
display(mc_aug["moment_specification_test_summary"].round(3))

print("Reference-minus-augmented moment distance comparison:")
display(mc_aug["moment_specification_comparison"].round(3))

Reference moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.126,2.239,0.606,0.002,0.065,0.009,1000,21,0.021,0.005,0.014,0.032,3.0,200,4
1,cov_identity,19.387,203.094,0.000,0.072,1.653,0.000,1000,1000,1.000,0.000,0.996,1.000,6.0,200,4


Augmented moment-specification test summary:


,test,distance,stat,p_value,mc_se_distance,mc_se_stat,mc_se_p_value,n_replications,n_rejections,reject_rate,reject_rate_mc_se,reject_ci_low,reject_ci_high,df,sample_size,bandwidth
0,mean_zero_hac,0.128,2.246,0.605,0.002,0.066,0.009,1000,20,0.02,0.004,0.013,0.031,3.0,200,4
1,cov_identity,19.285,200.298,0.000,0.070,1.602,0.000,1000,1000,1.00,0.000,0.996,1.000,6.0,200,4


Reference-minus-augmented moment distance comparison:


,test,n_replications,distance_ref,mc_se_distance_ref,distance_aug,mc_se_distance_aug,distance_improvement,mc_se_distance_improvement,stat_ref,mc_se_stat_ref,stat_aug,mc_se_stat_aug,stat_improvement,mc_se_stat_improvement,aug_closer_rate,aug_closer_rate_mc_se,aug_closer_ci_low,aug_closer_ci_high
0,mean_zero_hac,1000,0.126,0.002,0.128,0.002,-0.001,0.000,2.239,0.065,2.246,0.066,-0.007,0.002,0.300,0.014,0.272,0.329
1,cov_identity,1000,19.387,0.072,19.285,0.070,0.102,0.004,203.094,1.653,200.298,1.602,2.795,0.292,0.792,0.013,0.766,0.816
